# Chess Transformer Player — Step 2: Model Training

Fine-tunes **GPT-2 medium** (355M params) on chess positions using:
- **Loss masking**: only learns to predict the move, not the FEN string
- **Mixed precision (fp16)**: faster training on GPU
- **Fused AdamW**: more memory-efficient optimizer

**Run on**: Kaggle with GPU P100 (free, 12h sessions)

**Input**: `final_training_data.csv` uploaded as a Kaggle dataset  
**Output**: Fine-tuned model pushed to HuggingFace Hub

In [ ]:
# ── Install ───────────────────────────────────────────────────
!pip install transformers datasets accelerate huggingface_hub -q

In [ ]:
# ── Check GPU ─────────────────────────────────────────────────
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load and prepare training data ───────────────────────────
import pandas as pd

# Path when uploaded as Kaggle dataset
DATA_PATH = '/kaggle/input/chess-training-data/final_training_data.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows")
print(df['source'].value_counts())

# Subsample: all Stockfish annotations + 150k GM games
df_sf  = df[df['source'] == 'stockfish_annotated']                          # 200k high-quality
df_gm  = df[df['source'] == 'gm_elite'].sample(150_000, random_state=42)   # 150k GM games
df_train = pd.concat([df_sf, df_gm]).sample(frac=1, random_state=42).reset_index(drop=True)

# Format: GPT-2 sees FEN and learns to predict the move
df_train['text'] = "FEN: " + df_train['fen'] + " MOVE: " + df_train['move'] + "<|endoftext|>"

print(f"\nTraining examples: {len(df_train):,}")
print("Example:", df_train['text'].iloc[0])

In [ ]:
# ── Load tokenizer and tokenize dataset ──────────────────────
from transformers import GPT2Tokenizer
from datasets import Dataset

tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fast(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=96,
        padding='max_length',
    )

dataset  = Dataset.from_pandas(df_train[['text']])
tokenized = dataset.map(
    tokenize_fast,
    batched=True,
    batch_size=2000,
    remove_columns=['text'],
    num_proc=1
)
print(tokenized)

In [ ]:
# ── Custom loss-masking collator ──────────────────────────────
# Only computes loss on MOVE tokens, not FEN tokens.
# This makes training focus entirely on chess move prediction.
import torch
from transformers import DataCollatorForLanguageModeling

class MoveOnlyDataCollator:
    """Masks FEN tokens so loss is only computed on the MOVE part."""

    def __init__(self, tokenizer):
        self.tokenizer      = tokenizer
        self.base_collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        self.move_token_ids = tokenizer.encode(" MOVE:", add_special_tokens=False)
        print(f"MOVE token ids: {self.move_token_ids}")

    def __call__(self, features):
        batch     = self.base_collator(features)
        labels    = batch['labels'].clone()
        input_ids = batch['input_ids']

        for i in range(len(labels)):
            seq      = input_ids[i].tolist()
            move_pos = None
            for j in range(len(seq) - len(self.move_token_ids) + 1):
                if seq[j : j + len(self.move_token_ids)] == self.move_token_ids:
                    move_pos = j
                    break
            if move_pos is not None:
                labels[i, : move_pos + len(self.move_token_ids)] = -100

        batch['labels'] = labels
        return batch

data_collator = MoveOnlyDataCollator(tokenizer)

# Sanity check
sample     = [tokenized[i] for i in range(4)]
batch      = data_collator(sample)
n_unmasked = (batch['labels'] != -100).sum().item()
print(f"Unmasked label tokens in batch of 4: {n_unmasked}")
print("✓ Masking works" if n_unmasked < 50 else "⚠️ Masking may not be working")

In [ ]:
# ── Train ─────────────────────────────────────────────────────
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer

model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
print(f"Model parameters: {model.num_parameters():,}")

training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/chess_gpt2",
    num_train_epochs            = 3,
    per_device_train_batch_size = 32,
    gradient_accumulation_steps = 4,      # effective batch = 128
    learning_rate               = 5e-5,
    warmup_steps                = 200,
    save_steps                  = 1000,
    logging_steps               = 50,
    logging_first_step          = True,
    fp16                        = True,   # faster on GPU
    dataloader_pin_memory       = False,
    report_to                   = "none",
    save_total_limit            = 2,
    optim                       = "adamw_torch_fused",
    gradient_checkpointing      = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized,
    data_collator = data_collator,
)

trainer.train()

In [ ]:
# ── Save locally (insurance before HF push) ──────────────────
model.save_pretrained("/kaggle/working/chess_gpt2_final")
tokenizer.save_pretrained("/kaggle/working/chess_gpt2_final")
print("Saved locally ✓")

In [ ]:
# ── Sanity check: test the trained model ─────────────────────
!pip install python-chess -q
import chess, torch

def predict_move(fen, model, tokenizer, max_moves=30):
    """Score all legal moves and return the highest probability one."""
    board       = chess.Board(fen)
    legal_moves = [m.uci() for m in board.legal_moves]
    if not legal_moves:
        return None
    import random
    if len(legal_moves) > max_moves:
        legal_moves = random.sample(legal_moves, max_moves)

    model.eval()
    best_move, best_score = None, float('-inf')
    with torch.no_grad():
        for move in legal_moves:
            text   = f"FEN: {fen} MOVE: {move}"
            inputs = tokenizer(text, return_tensors='pt',
                               truncation=True, max_length=96).to(model.device)
            loss   = model(**inputs, labels=inputs['input_ids']).loss.item()
            if -loss > best_score:
                best_score = -loss
                best_move  = move
    return best_move

# Test on 3 positions
tests = [
    ("Starting position", "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"),
    ("After 1.e4",        "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"),
    ("Italian Game",      "r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4"),
]
for name, fen in tests:
    move = predict_move(fen, model, tokenizer)
    print(f"{name}: {move}")

In [ ]:
# ── Push to HuggingFace Hub ───────────────────────────────────
# Requires HF_TOKEN secret with WRITE access set in Kaggle Secrets
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

HF_REPO  = "Iman-ghotbi/chess-gpt2-finetuned"  # change to your username
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

# Create repo if it doesn't exist
api = HfApi()
api.create_repo(repo_id=HF_REPO, token=hf_token, exist_ok=True, private=False)
print("Repo ready ✓")

model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)
print(f"\nModel live at: https://huggingface.co/{HF_REPO}")